Анализ доступных индексов Московской биржи - какие следует взять в проект.

1. Получить индексы для выбранных 20 акций
   ↓
2. Посчитать покрытие наших акций индексами
   ↓
3. Выбрать рабочий набор индексов
   ↓
4. Проверить доступность дневной истории
   ↓
5. Проверить candleborders для 1 минуты
   ↓
6. Выгрузить тестовый период дневных данных
   ↓
7. Выгрузить тестовый период минутных свечей
   ↓
8. Зафиксировать итоговый список индексов

In [1]:
import pandas as pd
import requests
import time

pd.set_option('display.max_columns', None)

In [2]:
# Текущий список акций MOEX
selected_securities = [
    "SBER",
    "GAZP",
    "VTBR",
    "T",
    "LKOH",
    "NVTK",
    "ROSN",
    "GMKN",
    "YDEX",
    "PLZL",
    "SMLT",
    "OZON",
    "X5",
    "RUAL",
    "TATN",
    "AFKS",
    "MOEX",
    "PIKK",
    "POSI",
    "AQUA",
]

In [3]:
# Анализ вхождения акций в индексы MOEX

def get_index_list(security: str)-> pd.DataFrame:
    url = f'https://iss.moex.com//iss/securities/{security}/indices.json'
    response = requests.get(
        url=url,
        timeout=30
    )
    if response.status_code != 200:
        response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(
        columns=data['indices']['columns'],
        data=data['indices']['data']
    )
    df['security'] = security
    return df

index_moex_list = []
for security in selected_securities:
    index_moex_list.append(get_index_list(security))


df_index_moex = pd.concat(index_moex_list, ignore_index=True)

In [4]:
df_index_moex[df_index_moex['TILL'] == '2026-08-10'].drop_duplicates(subset=['security', 'SECID']).groupby(['SECID'])['security'].count().reset_index().sort_values('security', ascending=False)

,SECID,security
11,MOEXBMI,20
35,RUBMI,20
3,IMOEX,17
4,IMOEX2,17
0,EPSI,17
27,RTSI,17
5,IMOEXCNY,17
6,IMOEXW,17
20,MRRT,15
19,MRBC,14


In [5]:
# Выбор индексов и проверка по акциям
selected_indices = [
    "IMOEX",   # рынок в целом
    "MOEXOG",  # нефть и газ
    "MOEXFN",  # финансы
    "MOEXMM",  # металлы и добыча
    "MOEXIT",  # IT
]

df_index_moex[df_index_moex['SECID'].isin(selected_indices[1:])].sort_values(by="SECID")

,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,security
18,MOEXFN,Индекс финансов,2005-01-11,2026-08-10,8843.56,0.83,73.12,SBER
54,MOEXFN,Индекс финансов,2007-07-02,2026-08-10,8842.52,0.82,72.08,VTBR
68,MOEXFN,Индекс финансов,2024-11-28,2026-08-10,8842.52,0.82,72.08,T
299,MOEXFN,Индекс финансов,2013-06-18,2017-12-21,8842.52,0.82,72.08,AFKS
320,MOEXFN,Индекс финансов,2013-03-18,2026-08-10,8842.52,0.82,72.08,MOEX
160,MOEXIT,Индекс МосБиржи IT,2024-09-20,2026-08-10,1837.97,0.49,8.93,YDEX
211,MOEXIT,Индекс МосБиржи IT,2021-04-02,2026-08-10,1837.97,0.49,8.93,OZON
341,MOEXIT,Индекс МосБиржи IT,2022-09-16,2026-08-10,1837.97,0.49,8.93,POSI
153,MOEXMM,Индекс металлов и добычи,2005-01-11,2026-08-10,4732.08,0.62,29.13,GMKN
190,MOEXMM,Индекс металлов и добычи,2006-07-03,2026-08-10,4732.08,0.62,29.13,PLZL


In [6]:
# Получить список доступных данных для минутного и 24 часового интервала

def get_interval_candleborders(index) -> pd.DataFrame:
    url = f'https://iss.moex.com/iss/engines/stock/markets/index/securities/{index}/candleborders.json'
    response = requests.get(
        url=url,
        timeout=30
    )

    data = response.json()
    df = pd.DataFrame(
        columns=data['borders']['columns'],
        data=data['borders']['data']
    )
    df = df[df['interval'].isin([1, 24])]
    df = df.drop(columns = 'board_group_id')
    df['index'] = index
    return df

list_index_interval_candleborders: list[pd.DataFrame] = []
for moex_index in selected_indices:
    list_index_interval_candleborders.append(get_interval_candleborders(moex_index))

df_interval = pd.concat(list_index_interval_candleborders, ignore_index=True)
df_interval = df_interval.sort_values(by = 'interval')
df_interval

,begin,end,interval,index
0,2011-12-15 10:00:00,2026-08-11 10:59:59,1,IMOEX
2,2011-12-15 10:00:00,2026-08-11 10:59:59,1,MOEXOG
6,2011-12-15 10:00:00,2026-08-11 10:59:59,1,MOEXMM
4,2011-12-15 10:00:00,2026-08-11 10:59:59,1,MOEXFN
8,2025-07-14 09:50:00,2026-08-11 10:59:59,1,MOEXIT
3,2004-12-30 00:00:00,2026-08-11 10:57:12,24,MOEXOG
5,2004-12-30 00:00:00,2026-08-11 10:57:12,24,MOEXFN
1,1997-09-22 00:00:00,2026-08-11 10:57:12,24,IMOEX
7,2004-12-30 00:00:00,2026-08-11 10:57:12,24,MOEXMM
9,2020-12-25 00:00:00,2026-08-11 10:57:12,24,MOEXIT


In [7]:
index_interval_days = df_interval[df_interval['interval'] == 24].drop(columns= 'interval').set_index('index').to_dict('index')
index_interval_days

{'MOEXOG': {'begin': '2004-12-30 00:00:00', 'end': '2026-08-11 10:57:12'},
 'MOEXFN': {'begin': '2004-12-30 00:00:00', 'end': '2026-08-11 10:57:12'},
 'IMOEX': {'begin': '1997-09-22 00:00:00', 'end': '2026-08-11 10:57:12'},
 'MOEXMM': {'begin': '2004-12-30 00:00:00', 'end': '2026-08-11 10:57:12'},
 'MOEXIT': {'begin': '2020-12-25 00:00:00', 'end': '2026-08-11 10:57:12'}}

In [8]:
# Получить данные по индексам за период

def get_index_data_per_day(params: dict, url: str)-> tuple[pd.DataFrame, dict]:
    response = requests.get(
        url=url,
        timeout=30,
        params=params
    )

    if response.status_code != 200:
        response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(
        columns=data['history']['columns'],
        data=data['history']['data'],
    )
    cursor = dict(
                zip(
                    data['history.cursor']['columns'],
                    data['history.cursor']['data'][0]
                    ))
    return df, cursor


list_history_index: list[pd.DataFrame] = []
for index in index_interval_days:

    if index == 'MOEXIT':
        board = 'RTSI'
    else:
        board = 'SNDX'

    url = f'https://iss.moex.com/iss/history/engines/stock/markets/index/boards/{board}/securities/{index}.json'

    params = {
        'from': '2020-01-01',
        'till': index_interval_days[index]['end'],
        'start': 0
    }

    while True:
        _df, _step = get_index_data_per_day(params, url)
        list_history_index.append(_df)

        if _step['INDEX'] +  _step['PAGESIZE'] >= _step['TOTAL']:
            break

        params['start'] += _step['PAGESIZE']

        time.sleep(0.2)

df_history_index = pd.concat(list_history_index, ignore_index=True)

print(df_history_index.shape)
df_history_index.tail()

(8022, 20)


,BOARDID,SECID,TRADEDATE,SHORTNAME,NAME,CLOSE,OPEN,HIGH,LOW,VALUE,DURATION,YIELD,DECIMALS,CAPITALIZATION,CURRENCYID,DIVISOR,TRADINGSESSION,VOLUME,TRADE_SESSION_DATE,RECALC_DATE
8017,RTSI,MOEXIT,2026-08-04,Индекс МосБиржи IT,Индекс МосБиржи информационных технологий,1752.69,1742.59,1756.20,1726.68,9.230231e+09,0.0,0.0,2,1.277633e+11,RUB,7.289577e+07,3,14379988.0,2026-08-04,None
8018,RTSI,MOEXIT,2026-08-05,Индекс МосБиржи IT,Индекс МосБиржи информационных технологий,1806.61,1783.64,1807.50,1783.64,1.394000e+10,0.0,0.0,2,1.316941e+11,RUB,7.289577e+07,3,37595948.0,2026-08-05,None
8019,RTSI,MOEXIT,2026-08-06,Индекс МосБиржи IT,Индекс МосБиржи информационных технологий,1802.50,1813.67,1827.61,1790.33,1.211720e+10,0.0,0.0,2,1.313944e+11,RUB,7.289577e+07,3,39551273.0,2026-08-06,None
8020,RTSI,MOEXIT,2026-08-07,Индекс МосБиржи IT,Индекс МосБиржи информационных технологий,1796.06,1804.78,1816.67,1786.71,6.474199e+09,0.0,0.0,2,1.309251e+11,RUB,7.289577e+07,3,14652083.0,2026-08-07,None
8021,RTSI,MOEXIT,2026-08-10,Индекс МосБиржи IT,Индекс МосБиржи информационных технологий,1829.04,1813.84,1831.36,1804.06,1.143719e+10,0.0,0.0,2,1.333295e+11,RUB,7.289577e+07,3,16349608.0,2026-08-10,None


In [9]:
# Получить тестовые минутные свечи по выбранным индексам за карйнюю дату

index_interval_minutes = (
    df_interval[
        (df_interval["interval"] == 1)
        & (df_interval["index"].isin(selected_indices))
    ]
    .drop(columns="interval")
    .set_index("index")
    .to_dict("index")
)


def get_index_candles_1m(
    index: str,
    date: str,
    start: int
) -> pd.DataFrame:

    url = (
        f"https://iss.moex.com/iss/engines/stock/markets/index/securities/{index}/candles.json"
    )

    params = {
        "interval": 1,
        "from": date,
        "till": date,
        "start": start
    }

    response = requests.get(
        url=url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(
        columns=data["candles"]["columns"],
        data=data["candles"]["data"]
    )

    return df


# Берём последнюю дату, которая доступна для всех выбранных индексов
test_date = min(
    pd.to_datetime(data["end"]).date()
    for data in index_interval_minutes.values()
).isoformat()


list_index_candles_1m: list[pd.DataFrame] = []

for index in selected_indices:

    start = 0

    while True:

        df_step = get_index_candles_1m(
            index=index,
            date=test_date,
            start=start
        )

        if df_step.empty:
            break

        df_step["SECID"] = index

        list_index_candles_1m.append(df_step)

        # Максимальный размер страницы свечей — 500 строк
        if len(df_step) < 500:
            break

        start += 500

        time.sleep(0.2)


df_index_candles_1m = pd.concat(
    list_index_candles_1m,
    ignore_index=True
)

df_index_candles_1m

,open,close,high,low,value,volume,begin,end,SECID
0,2320.13,2320.14,2320.25,2319.82,1.281311e+08,0,2026-08-11 09:50:00,2026-08-11 09:50:59,IMOEX
1,2320.13,2319.56,2320.60,2319.38,1.935190e+08,0,2026-08-11 09:51:00,2026-08-11 09:51:59,IMOEX
2,2319.59,2318.35,2319.59,2318.26,2.336770e+08,0,2026-08-11 09:52:00,2026-08-11 09:52:59,IMOEX
3,2318.37,2318.60,2318.60,2318.06,1.068496e+08,0,2026-08-11 09:53:00,2026-08-11 09:53:59,IMOEX
4,2318.60,2318.35,2318.83,2317.99,8.887926e+07,0,2026-08-11 09:54:00,2026-08-11 09:54:59,IMOEX
...,...,...,...,...,...,...,...,...,...
340,1840.39,1840.30,1840.55,1840.30,1.976127e+06,0,2026-08-11 10:54:00,2026-08-11 10:54:59,MOEXIT
341,1839.62,1839.60,1840.17,1839.60,9.924663e+06,0,2026-08-11 10:55:00,2026-08-11 10:55:59,MOEXIT
342,1838.45,1837.55,1838.64,1837.55,3.390111e+07,0,2026-08-11 10:56:00,2026-08-11 10:56:59,MOEXIT
343,1837.97,1837.92,1837.97,1837.27,2.133114e+07,0,2026-08-11 10:57:00,2026-08-11 10:57:59,MOEXIT


In [10]:
df_index_candles_1m.groupby("SECID").agg(
    rows=("begin", "count"),
    begin=("begin", "min"),
    end=("end", "max")
)
df_index_candles_1m.columns

Index(['open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end',
       'SECID'],
      dtype='str')